# 20 — Coastal Gradient Experiment

Replacement notebook with retry logic and caching.

In [ ]:

SITES = [
    {"site_id": "EST", "site_name": "Eastbourne",   "lat": 50.7680,  "lon": 0.2900},
    {"site_id": "NHV", "site_name": "Newhaven ERF", "lat": 50.80208, "lon": 0.04961},
    {"site_id": "LWS", "site_name": "Lewes",        "lat": 50.8739,  "lon": 0.0110},
]
DATE_FROM = "2024-12-26"
DATE_TO   = "2024-12-31"
PRODUCT_KEY = "NO2"
TIMELINESS = "OFFL"
SCENE_CATALOG_DIR = "outputs/s5p"
OUTPUT_DIR = "outputs/gradient"
MAX_TIME_MISMATCH_HOURS = 3


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_vars,
        "timezone": "UTC",
    }
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(OUTPUT_DIR) / "weather_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

catalog_rows = []
for site in SITES:
    p = Path(SCENE_CATALOG_DIR) / f"{site['site_id']}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.csv"
    if p.exists():
        df = pd.read_csv(p)
        df["site_id"] = site["site_id"]
        df["site_name"] = site["site_name"]
        catalog_rows.append(df)
cat = pd.concat(catalog_rows, ignore_index=True) if catalog_rows else pd.DataFrame()
if not cat.empty:
    cat["start_utc"] = pd.to_datetime(cat["start_utc"], utc=True, errors="coerce")
    cat["date"] = cat["start_utc"].dt.date.astype("string")
    cat = cat[(cat["start_utc"] >= pd.Timestamp(DATE_FROM, tz="UTC")) &
              (cat["start_utc"] <= pd.Timestamp(DATE_TO, tz="UTC") + pd.Timedelta(days=1))]
display(cat.head(10))


In [ ]:

def fetch_openmeteo_hourly(lat, lon, start_date, end_date):
    j = fetch_weather_cached(CACHE_DIR, lat, lon, start_date, end_date, "wind_speed_10m,wind_direction_10m,cloud_cover")
    h = j.get("hourly", {})
    return pd.DataFrame({
        "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
        "wind_speed_ms": h.get("wind_speed_10m", []),
        "wind_dir_deg": h.get("wind_direction_10m", []),
        "cloud_cover_pct": h.get("cloud_cover", []),
    })
weather_rows = []
for site in SITES:
    w = fetch_openmeteo_hourly(site["lat"], site["lon"], DATE_FROM, DATE_TO)
    w["site_id"] = site["site_id"]
    w["site_name"] = site["site_name"]
    weather_rows.append(w)
weather = pd.concat(weather_rows, ignore_index=True)

def nearest_hour(ts, w):
    if w.empty or pd.isna(ts):
        return np.nan, np.nan, np.nan, np.nan
    idx = (w["time_utc"] - ts).abs().idxmin()
    row = w.loc[idx]
    dt_h = abs((row["time_utc"] - ts).total_seconds()) / 3600.0
    return row["wind_speed_ms"], row["wind_dir_deg"], row["cloud_cover_pct"], dt_h

rows = []
for _, r in cat.iterrows():
    w = weather[weather["site_id"] == r["site_id"]].copy()
    ws, wd, cc, dt_h = nearest_hour(r["start_utc"], w)
    if pd.notna(dt_h) and dt_h > MAX_TIME_MISMATCH_HOURS:
        continue
    rows.append({"date": str(r["date"]), "site_id": r["site_id"], "site_name": r["site_name"], "start_utc": r["start_utc"], "scene_name": r.get("name"), "product_id": r.get("product_id"), "wind_speed_ms": ws, "wind_dir_deg": wd, "cloud_cover_pct": cc})
gradient = pd.DataFrame(rows)
distance_order = {"EST": 0, "NHV": 1, "LWS": 2}
if not gradient.empty:
    gradient["position_index"] = gradient["site_id"].map(distance_order)
    daily_summary = gradient.groupby(["date","site_id","site_name","position_index"], dropna=False).agg(mean_wind_speed_ms=("wind_speed_ms","mean"), mean_cloud_cover_pct=("cloud_cover_pct","mean"), scenes=("product_id","nunique")).reset_index().sort_values(["date","position_index"])
else:
    daily_summary = pd.DataFrame()
gradient.to_csv(Path(OUTPUT_DIR) / "coastal_gradient_candidate_table.csv", index=False)
daily_summary.to_csv(Path(OUTPUT_DIR) / "coastal_gradient_daily_summary.csv", index=False)
display(daily_summary.head(20))


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
if not daily_summary.empty:
    for site in ["EST","NHV","LWS"]:
        s = daily_summary[daily_summary["site_id"] == site]
        if not s.empty:
            ax.plot(pd.to_datetime(s["date"]), s["scenes"], marker="o", label=site)
ax.set_title("Coastal gradient experiment")
ax.set_xlabel("Date")
ax.set_ylabel("Scene count")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plot_path = Path(OUTPUT_DIR) / "coastal_gradient_experiment_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()
(Path(OUTPUT_DIR) / "coastal_gradient_manifest.json").write_text(json.dumps({"scene_catalog_rows": int(len(cat)), "gradient_rows": int(len(gradient)), "daily_summary_rows": int(len(daily_summary))}, indent=2), encoding="utf-8")
print("Saved:", plot_path)
